# Tutorial 6: Mixed Precision Quantization Search with Mase and Optuna

In this tutorial, we'll see how Mase can be integrated with Optuna, the popular hyperparameter optimization framework, to search for a Bert model optimized for sequence classification on the IMDb dataset. We'll take the Optuna-generated model and import it into Mase, then run the CompressionPipeline to prepare the model for edge deployment by quantizing and pruning its weights.

As we'll see, running Architecture Search with Mase/Optuna involves the following steps.

1. **Define the search space**: this is a dictionary containing the range of values for each parameter at each layer in the model.

2. **Write the model constructor**: this is a function which uses Optuna utilities to sample a model from the search space, and constructs the model using transformers from_config class method.

3. **Write the objective function**: this function calls on the model constructor defined in Step 2 and defines the training/evaluation setup for each search iteration.

4. **Go!** Choose an Optuna sampler, create a study and launch the search.

# Quantizers

## Software-simulated quantization

| name | paper | model@dataset | blocking dimension | representation | extra |
| --- | --- | --- | --- | --- | --- |
| `integer_quantizer(x, width, frac_width)` | - | - | - | $(i/2^{f})$, <br> signed int $i$, the number of fractional bits $f$ | signed fixed-point number |
| `minifloat_denorm_quantizer(x, width, exponent_width, exponent_bias)` | - | - | - | $(-1)^s 2^e m$, <br> exponent $e$, mantissa $m$ | no implicit leading bit in mantissa|
| `minifloat_ieee_quantizer(x, width, exponent_width, exponent_bias)` | - | - | - | $(-1)^s 2^e m'$, <br> normal: $m'=1.0+m$, subnormal: $m'=m$ | an implicit leading bit in mantissa |
| `log_quantizer(x, width, exponent_bias)` | [CNNs using Logarithmic Data Representation](http://arxiv.org/abs/1603.01025) | VGG16@CIFAR10, ALEXNET@CIFAR10 | - | $(-1)^s 2^e$ | - |
| `msfp_quatizer(x, width, exponent_width, exponent_bias, block_size)`| [Microsoft MSFP](https://proceedings.neurips.cc/paper/2020/hash/747e32ab0fea7fbd2ad9ec03daa3f840-Abstract.html) | CNNs, RNNs, <br> Transformers (BERT@MRPC, BERT@SQuAD1.1, BERT@SQuADv2) | Linear matrix: tiles along matrix row, <br> Conv2D: tiles along channel depth | $2^{e_{shared}}[(-1)^{s_1} m_1, (-1)^{s_2} m_2, \dots]$ |
| `block_minifloat_quantizer(x, width, exponent_width, bias_width, block_size)` | [Philip Leong's Block Minifloat](https://openreview.net/forum?id=6zaTwpNSsQ2) | CNNs, RNNs, Transformer (Transformer-base@IWSLT) | Matrix Multiply: $N\times N$ square block. <br> Conv2D (?) | $2^{-b_{shared}}[(-1)^{s_1} 2^{e'_1}m'_1, (-1)^{s_2}2^{e'_2}m'_2, \dots]$,  <br> the shared exponent bias:$b_{shared}$|  both forward and backward uses software-simulated quantized values|
| `block_log_quantizer(x, width, exponent_bias_width, block_size)` | - | - | - | $2^{-b_{shared}}[(-1)^{s_1} 2^{e'_1}, (-1)^{s_2}2^{e'_2}, \dots]$, <br> the shared exponent bias $b_{shared}$ |  |

The following quantizers are not supported yet

| name | paper | model@dataset | blocking dimension | representation | extra |
| --- | --- | --- | --- | --- | --- |
| ⬜ TODO: `mx_quantizer` | [Microsoft's MX](https://arxiv.org/abs/2302.08007) | See Table III in the paper. CNNs, RNNs, ViT (DeiT-Tiny/-Small@ImageNet), <br> Transformer (Transformer-base/-large@WMT-17, BERT-base/-large@Wikipedia, GPT-XS/-S/-M/-L/-XL@?) | Two-level scaling on vectors | $2^{e_{s}}\Big[ 2^{e_{ss_1}} [(-1)^{s_1} m_1, (-1)^{s_2} m_2  ], 2^{e_{ss_2}}[(-1)^{s_3} m_3, (-1)^{s_3} m_3 ]\Big]$ | no implicit leading bit in mantissa  |

## Two-level block quantization for large language models

Large language models requires significant GPU resources for inference. One way to reduce the inference resource consumption is quantization. The challenge of quantizing models is the significant accuracy degradation as the bit-width decreases. To solve this accuracy degradation, block-based number formats have been proposed. One or two levels of scaling factors are shared over a block of numbers, where the scaling factor can be exponent, exponent bias, mantissa, fixed-point number, or floating-point number. A proper blocking and sharing scheme mitigates the impact of extreme outlier values. However, block-based quantization of large language models remains to be explored, especially low bit width (1-bit/2-bit) quantization. Here we aim to explore different combinations of shared components on large language models. Specifically, we aim to estimate the hardware cost of each combination, and compare corresponding accuracy degradation given the same block size.

💡In `Facebook/OPT-350m` for language modeling, the Linear layers take up `168.02G / 174.68G x 100%=96.19%` FLOPs.

In [25]:
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

## Importing the model

If you are starting from scratch, you can load the Bert checkpoint directly from HuggingFace.

In [26]:
from transformers import AutoModel

model = AutoModel.from_pretrained(checkpoint)

If you have previously ran the tutorial on Neural Architecture Search (NAS), run the following cell to import the best model obtained from the search process.

In [27]:
from pathlib import Path
import dill

with open(f"{Path.home()}/mase_hh1425/tutorial_5_best_model.pkl", "rb") as f:
    base_model = dill.load(f)

First, fetch the dataset using the `get_tokenized_dataset` utility.

In [28]:
from chop.tools import get_tokenized_dataset

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

INFO     Tokenizing dataset imdb with AutoTokenizer for bert-base-uncased.


## 1. Defining the Search Space

We'll start by defining a search space, i.e. enumerating the possible combinations of hyperparameters that Optuna can choose during search. We'll explore the following range of values for the model's hidden size, intermediate size, number of layers and number of heads.

In [29]:
# Seacrh space provided by leon
 
import torch
from chop.nn.quantized.modules.linear import (
    LinearInteger, # check
    LinearMinifloatDenorm, # check
    LinearMinifloatIEEE, # check
    LinearLog, # check
    LinearBlockFP, # check  
    LinearBlockMinifloat, # check
    LinearBlockLog,  # check
    LinearBinary, # check
    LinearBinaryScaling, 
    #LinearBinaryResidualSign, not supported
    LinearTernary, # check
)
 
search_space = {
    "linear_layer_choices": [
        torch.nn.Linear,
        LinearInteger, # check
        LinearMinifloatDenorm, # check
        LinearMinifloatIEEE, # check
        LinearLog, # check
        LinearBlockFP, # check
        LinearBlockMinifloat, # check
        LinearBlockLog,# check
        LinearBinary, # check
        LinearBinaryScaling, # check
        LinearTernary,
    ],
    "int_width_choices": [8, 16, 32],
    "int_frac_choices": [2, 4, 8],
    "block_size_choices": [8, 16, 32],
    "exp_size_choices": [4, 8, 16],
    "bool_choices": [True, False],
}
 
import torch
 
def get_tensor_stats(tensor):
 
    return {
 
        "mean": tensor.mean().item(),
 
        "median": tensor.median().item(),
 
        "max": tensor.max().item(),
 
    } # For any precisions needing stats

## 2. Writing a Model Constructor

We define the following function, which will get called in each iteration of the search process. The function is passed the `trial` argument, which is an Optuna object that comes with many functionalities - see the [Trial documentation](https://optuna.readthedocs.io/en/stable/reference/trial.html) for more details. Here, we use the `trial.suggest_categorical` function, which triggers the chosen sampler to choose a layer type. The suggested integer is the index into the search space for each parameter, which we defined in the previous cell.

In [ ]:
from chop.tools.utils import deepsetattr
from copy import deepcopy
 
 
def construct_model(trial, set_precision = None):
 
    # Fetch the model
    trial_model = deepcopy(base_model)

 
    # Quantize layers according to optuna suggestions
    for name, layer in trial_model.named_modules():
        if isinstance(layer, torch.nn.Linear):
 
            w_stats = get_tensor_stats(layer.weight)
            b_stats = get_tensor_stats(layer.bias)
 
            if set_precision == None:
            # Per Layer Trials
                new_layer_cls = trial.suggest_categorical(
                    f"{name}_type",
                    search_space["linear_layer_choices"],
                )
            elif set_precision in search_space["linear_layer_choices"]:   
            # Global Layer Trials Per precision
                new_layer_cls = set_precision
 
            if new_layer_cls == torch.nn.Linear:
                continue
 
            kwargs = {
                "in_features": layer.in_features,
                "out_features": layer.out_features,
            }
 
            # If the chosen layer is integer, define the low precision config
            if new_layer_cls == LinearInteger:
                w = trial.suggest_categorical(f"{name}_width", search_space["int_width_choices"])
                f = trial.suggest_categorical(f"{name}_frac",  search_space["int_frac_choices"])
                kwargs["config"] = {
                    "data_in_width": w,
                    "data_in_frac_width": f,
                    "weight_width": w,
                    "weight_frac_width": f,
                    "bias_width": w,
                    "bias_frac_width": f,
                }
 
            elif new_layer_cls in [LinearMinifloatDenorm, LinearMinifloatIEEE]:
                w = trial.suggest_categorical(f"{name}_width", search_space["int_width_choices"])
                e = trial.suggest_categorical(f"{name}_exp", search_space["exp_size_choices"]) #
                kwargs["config"] = {
                    "data_in_width": w,
                    "data_in_exponent_width": e,
                    "data_in_exponent_bias": None,
                    "weight_width": w,
                    "weight_exponent_width": e,
                    "weight_exponent_bias": None,
                    "bias_width": w,
                    "bias_exponent_width": e,
                    "bias_exponent_bias": None,
                }

            elif new_layer_cls == LinearLog:
                w = trial.suggest_categorical(f"{name}_bfp_w", search_space["int_width_choices"])
                e = trial.suggest_categorical(f"{name}_bfp_e", search_space["exp_size_choices"])
 
                kwargs["config"] = {
                    "data_in_width": w, 
                    "data_in_exponent_bias": None,

                    "weight_width": w,
                    "weight_exponent_bias": None,
 
                    "bias_width": w,
                    "bias_exponent_bias": None,
                }

            elif new_layer_cls == LinearBlockFP:
                w = trial.suggest_categorical(f"{name}_bfp_w", search_space["int_width_choices"])
                bs = trial.suggest_categorical(f"{name}_bfp_bs", search_space["block_size_choices"])
                e = trial.suggest_categorical(f"{name}_bfp_e", search_space["exp_size_choices"])
 
                kwargs["config"] = {
                    "data_in_width": w, 
                    "data_in_block_size": [bs],
                    "data_in_exponent_width": e, 
                    "data_in_exponent_bias": None,
 
                    "weight_width": w, 
                    "weight_block_size": [bs],
                    "weight_exponent_width": e, 
                    "weight_exponent_bias": None,
 
                    "bias_width": w,
                    "bias_exponent_width": e,
                    "bias_exponent_bias": None,
                    "bias_block_size": [bs],
                }
 
            elif new_layer_cls == LinearBlockMinifloat:
                w = trial.suggest_categorical(f"{name}_bw", search_space["int_width_choices"])
                bs = trial.suggest_categorical(f"{name}_bs", search_space["block_size_choices"])
                e = trial.suggest_categorical(f"{name}_be", search_space["exp_size_choices"])
 
                kwargs["config"] = {
                    "data_in_width": w,
                    "data_in_block_size": [bs],
                    "data_in_exponent_width": e,
                    "data_in_exponent_bias_width": e,
 
                    "weight_width": w,
                    "weight_block_size": [bs],
                    "weight_exponent_width": e,
                    "weight_exponent_bias_width": e,
 
                    "bias_width": w,
                    "bias_exponent_width": e,
                    "bias_exponent_bias_width": e,
                    "bias_block_size": [bs],
                }
 
            elif new_layer_cls == LinearBlockLog:
                w = trial.suggest_categorical(f"{name}_bl_w", search_space["int_width_choices"])
                bs = trial.suggest_categorical(f"{name}_bl_bs", search_space["block_size_choices"])
                e = trial.suggest_categorical(f"{name}_bl_e", search_space["exp_size_choices"])
               
                kwargs["config"] = {
                    # Input
                    "data_in_width": w,
                    "data_in_block_size": [bs],
                    "data_in_exponent_bias_width": e,
 
                    # Weight
                    "weight_width": w,
                    "weight_block_size": [bs],
                    "weight_exponent_bias_width": e,
 
                    "bias_width": w,
                    "bias_exponent_bias_width": e,
                    "bias_block_size": [bs],
                }
 
            elif new_layer_cls == LinearBinary:
                is_bipolar = trial.suggest_categorical(f"{name}_bipolar", search_space["bool_choices"])
                w = trial.suggest_categorical(f"{name}_width", search_space["int_width_choices"])
                f = trial.suggest_categorical(f"{name}_frac", search_space["int_frac_choices"])
                kwargs["config"] = {
                    "weight_bipolar": is_bipolar,
                    "weight_stochastic": not is_bipolar,
                }
 
 
            elif new_layer_cls == LinearBinaryScaling:
                is_bipolar = trial.suggest_categorical(f"{name}_bipolar", search_space["bool_choices"])
                kwargs["config"] = {
                    "data_in_bipolar": is_bipolar,
                    "weight_bipolar": is_bipolar, 
                    "bias_bipolar": is_bipolar,
                    "data_in_stochastic": not is_bipolar,
                    "weight_stochastic": not is_bipolar,
                    "bias_stochastic": not is_bipolar,
                    
                    "binary_training": True,

                }
 
            elif new_layer_cls == LinearTernary:
                w = trial.suggest_categorical(f"{name}_width", search_space["int_width_choices"])
                f = trial.suggest_categorical(f"{name}_frac", search_space["int_frac_choices"])
                sf = trial.suggest_categorical(f"{name}_scaling", search_space["bool_choices"])
                kwargs["config"] = {
                    "weight_scaling_factor": sf,
                    "weight_mean": w_stats["mean"],
                    "weight_median": w_stats["median"],
                    "weight_max": w_stats["max"],
                }

            # Create the new layer (copy the weights)
            new_layer = new_layer_cls(**kwargs)
 
            new_layer.weight.data = layer.weight.data
 
            # Replace the layer in the model
            deepsetattr(trial_model, name, new_layer)
 
    return trial_model

## 3. Defining the Objective Function

Next, we define the objective function for the search, which gets called on each trial. In each trial, we create a new model instace with chosen hyperparameters according to the defined sampler. We then use the `get_trainer` utility in Mase to run a training loop on the IMDb dataset for a number of epochs. Finally, we use `evaluate` to report back the classification accuracy on the test split.

In [35]:
from chop.tools import get_trainer
import random


def objective(trial, set_precision=None):

    # Define the model
    model = construct_model(trial, set_precision=set_precision)

    trainer = get_trainer(
        model=model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=1,
    )

    trainer.train()
    eval_results = trainer.evaluate()

    trial.set_user_attr("model", model)

    if set_precision is not None:
        precision = set_precision.__name__
    else:
        precision = "Mixed"
        
    trial.set_user_attr("model_type", precision)

    return eval_results["eval_accuracy"]

## 4. Launching the Search

Optuna provides a number of samplers, for example:

* **GridSampler**: iterates through every possible combination of hyperparameters in the search space
* **RandomSampler**: chooses a random combination of hyperparameters in each iteration
* **TPESampler**: uses Tree-structured Parzen Estimator algorithm to choose hyperparameter values.

You can define the chosen sampler by simply importing from `optuna.samplers` as below.

In [32]:
from optuna.samplers import GridSampler, RandomSampler, TPESampler

sampler = TPESampler()

With all the pieces in place, we can launch the search as follows. The number of trials is set to 1 so you can go get a coffee for 10 minutes, then proceed with the tutorial. However, this will essentially be a random model - for better results, set this to 100 and leave it running overnight!

In [ ]:
import optuna


precisions_to_test = [
    LinearMinifloatIEEE, 
    LinearLog, 
    LinearBlockFP, 
    LinearBlockMinifloat, 
    LinearBlockLog,
    LinearBinary, 
    LinearBinaryScaling, 
    LinearTernary
]

for precision_type in precisions_to_test:

    precision_name = precision_type.__name__
    print(f"\n ========Starting Study for Global Precision: {precision_name}============")

    study = optuna.create_study(
        direction="maximize",
        study_name=f"bert-tiny-nas-study-{precision_name}",
        sampler=sampler,
    )

    study.optimize(
        lambda trial: objective(trial, set_precision=precision_type),
        n_trials=30,
        timeout=60 * 60 * 24,
    )

    best_acc_history = []
    current_best = 0

    for trial in study.trials:
        if trial.value > current_best:
            current_best = trial.value
        best_acc_history.append(current_best)

    print("Global best Aacc history:", best_acc_history)
    print("Best params:", study.best_params)
    print("=================================================================================")



[I 2026-02-04 22:48:30,294] A new study created in memory with name: bert-tiny-nas-study
/home/testunot/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.365100
1000,0.321400
1500,0.310600
2000,0.317200
2500,0.318600
3000,0.328900


[I 2026-02-04 22:52:32,922] Trial 0 finished with value: 0.86092 and parameters: {'bert.encoder.layer.0.attention.self.query_width': 8, 'bert.encoder.layer.0.attention.self.query_exp': 8, 'bert.encoder.layer.0.attention.self.key_width': 32, 'bert.encoder.layer.0.attention.self.key_exp': 16, 'bert.encoder.layer.0.attention.output.dense_width': 16, 'bert.encoder.layer.0.attention.output.dense_exp': 8, 'bert.encoder.layer.0.intermediate.dense_width': 32, 'bert.encoder.layer.0.intermediate.dense_exp': 8, 'bert.encoder.layer.0.output.dense_width': 16, 'bert.encoder.layer.0.output.dense_exp': 4, 'bert.encoder.layer.1.attention.self.key_width': 32, 'bert.encoder.layer.1.attention.self.key_exp': 8, 'bert.encoder.layer.1.attention.output.dense_width': 8, 'bert.encoder.layer.1.attention.output.dense_exp': 4, 'bert.encoder.layer.1.intermediate.dense_width': 8, 'bert.encoder.layer.1.intermediate.dense_exp': 16, 'bert.encoder.layer.1.output.dense_width': 8, 'bert.encoder.layer.1.output.dense_exp': 

Step,Training Loss
500,14.954300
1000,29.033700
1500,41.325300
2000,51.589300
2500,57.843100
3000,58.056200


[I 2026-02-04 22:57:15,482] Trial 1 finished with value: 0.5 and parameters: {'bert.encoder.layer.0.attention.self.query_width': 32, 'bert.encoder.layer.0.attention.self.query_exp': 16, 'bert.encoder.layer.0.attention.self.key_width': 8, 'bert.encoder.layer.0.attention.self.key_exp': 4, 'bert.encoder.layer.0.attention.output.dense_width': 8, 'bert.encoder.layer.0.attention.output.dense_exp': 4, 'bert.encoder.layer.0.intermediate.dense_width': 16, 'bert.encoder.layer.0.intermediate.dense_exp': 16, 'bert.encoder.layer.0.output.dense_width': 8, 'bert.encoder.layer.0.output.dense_exp': 8, 'bert.encoder.layer.1.attention.self.key_width': 8, 'bert.encoder.layer.1.attention.self.key_exp': 8, 'bert.encoder.layer.1.attention.output.dense_width': 16, 'bert.encoder.layer.1.attention.output.dense_exp': 4, 'bert.encoder.layer.1.intermediate.dense_width': 16, 'bert.encoder.layer.1.intermediate.dense_exp': 4, 'bert.encoder.layer.1.output.dense_width': 32, 'bert.encoder.layer.1.output.dense_exp': 8, '

...

[I 2026-02-05 00:56:38,612] Trial 27 finished with value: 0.87664 and parameters: {'bert.encoder.layer.0.attention.self.query_width': 16, 'bert.encoder.layer.0.attention.self.query_exp': 4, 'bert.encoder.layer.0.attention.self.key_width': 8, 'bert.encoder.layer.0.attention.self.key_exp': 4, 'bert.encoder.layer.0.attention.output.dense_width': 32, 'bert.encoder.layer.0.attention.output.dense_exp': 16, 'bert.encoder.layer.0.intermediate.dense_width': 8, 'bert.encoder.layer.0.intermediate.dense_exp': 4, 'bert.encoder.layer.0.output.dense_width': 8, 'bert.encoder.layer.0.output.dense_exp': 16, 'bert.encoder.layer.1.attention.self.key_width': 16, 'bert.encoder.layer.1.attention.self.key_exp': 16, 'bert.encoder.layer.1.attention.output.dense_width': 16, 'bert.encoder.layer.1.attention.output.dense_exp': 16, 'bert.encoder.layer.1.intermediate.dense_width': 32, 'bert.encoder.layer.1.intermediate.dense_exp': 8, 'bert.encoder.layer.1.output.dense_width': 32, 'bert.encoder.layer.1.output.dense_ex

Step,Training Loss
500,0.369600
1000,0.301800
1500,0.286700
2000,0.297800
2500,0.282500
3000,0.311700


[I 2026-02-05 01:00:59,491] Trial 28 finished with value: 0.87628 and parameters: {'bert.encoder.layer.0.attention.self.query_width': 16, 'bert.encoder.layer.0.attention.self.query_exp': 4, 'bert.encoder.layer.0.attention.self.key_width': 8, 'bert.encoder.layer.0.attention.self.key_exp': 4, 'bert.encoder.layer.0.attention.output.dense_width': 32, 'bert.encoder.layer.0.attention.output.dense_exp': 16, 'bert.encoder.layer.0.intermediate.dense_width': 8, 'bert.encoder.layer.0.intermediate.dense_exp': 4, 'bert.encoder.layer.0.output.dense_width': 8, 'bert.encoder.layer.0.output.dense_exp': 16, 'bert.encoder.layer.1.attention.self.key_width': 16, 'bert.encoder.layer.1.attention.self.key_exp': 16, 'bert.encoder.layer.1.attention.output.dense_width': 16, 'bert.encoder.layer.1.attention.output.dense_exp': 16, 'bert.encoder.layer.1.intermediate.dense_width': 32, 'bert.encoder.layer.1.intermediate.dense_exp': 8, 'bert.encoder.layer.1.output.dense_width': 32, 'bert.encoder.layer.1.output.dense_ex

Step,Training Loss
500,0.368500
1000,0.301800
1500,0.283900
2000,0.288900
2500,0.280000
3000,0.312800


[I 2026-02-05 01:05:35,426] Trial 29 finished with value: 0.878 and parameters: {'bert.encoder.layer.0.attention.self.query_width': 16, 'bert.encoder.layer.0.attention.self.query_exp': 4, 'bert.encoder.layer.0.attention.self.key_width': 8, 'bert.encoder.layer.0.attention.self.key_exp': 4, 'bert.encoder.layer.0.attention.output.dense_width': 32, 'bert.encoder.layer.0.attention.output.dense_exp': 16, 'bert.encoder.layer.0.intermediate.dense_width': 8, 'bert.encoder.layer.0.intermediate.dense_exp': 4, 'bert.encoder.layer.0.output.dense_width': 8, 'bert.encoder.layer.0.output.dense_exp': 16, 'bert.encoder.layer.1.attention.self.key_width': 16, 'bert.encoder.layer.1.attention.self.key_exp': 16, 'bert.encoder.layer.1.attention.output.dense_width': 16, 'bert.encoder.layer.1.attention.output.dense_exp': 16, 'bert.encoder.layer.1.intermediate.dense_width': 32, 'bert.encoder.layer.1.intermediate.dense_exp': 8, 'bert.encoder.layer.1.output.dense_width': 32, 'bert.encoder.layer.1.output.dense_exp'

In [34]:
best_acc_history = []
current_best = 0

for trial in study.trials:
    if trial.value > current_best:
        current_best = trial.value
    best_acc_history.append(current_best)

print("Global best Aacc history:", best_acc_history)
print("Best params:", study.best_params)

Global best Aacc history: [0.86092, 0.86092, 0.86092, 0.86092, 0.86092, 0.86092, 0.86092, 0.86092, 0.86092, 0.86352, 0.86352, 0.86352, 0.86352, 0.86352, 0.86352, 0.86352, 0.86352, 0.86352, 0.86352, 0.87288, 0.87564, 0.87636, 0.87636, 0.87704, 0.87704, 0.87704, 0.87736, 0.87736, 0.87736, 0.878]
Best params: {'bert.encoder.layer.0.attention.self.query_width': 16, 'bert.encoder.layer.0.attention.self.query_exp': 4, 'bert.encoder.layer.0.attention.self.key_width': 8, 'bert.encoder.layer.0.attention.self.key_exp': 4, 'bert.encoder.layer.0.attention.output.dense_width': 32, 'bert.encoder.layer.0.attention.output.dense_exp': 16, 'bert.encoder.layer.0.intermediate.dense_width': 8, 'bert.encoder.layer.0.intermediate.dense_exp': 4, 'bert.encoder.layer.0.output.dense_width': 8, 'bert.encoder.layer.0.output.dense_exp': 16, 'bert.encoder.layer.1.attention.self.key_width': 16, 'bert.encoder.layer.1.attention.self.key_exp': 16, 'bert.encoder.layer.1.attention.output.dense_width': 16, 'bert.encoder.la